*  DSC530-T301 Data Exploration and Analysis
*  Weeks 3 & 4 Coding Assignment
*  Ander (Trevor) McKee

# Hands-on data analysis with Pandas

### Environment Setup Note

I've been doing the assignments using the environment specified by the textbook, Python 3.8.

Like you mentioned in the material, a few functions added to Python's standard library in 3.10+ are not available to me — statistics.covariance() and statistics.correlation(), which are used in Exercise 8. I substituted the equivalent NumPy functions (np.cov() and np.corrcoef()), which produce the same results and run on Python 3.8.

## Chapter 1, Exercise 4

In [1]:
import random

# Seed the generator so the 100 values are reproducible across runs
random.seed(0)

# 100 random salaries from 0-1,000,000, each rounded to the nearest thousand
salaries = [round(random.random()*1000000, -3) for _ in range(100)]

> **My note:** I kept the provided `random.seed(0)` so I can check my answers against the textbook. Like instructed, I'm treating these values as a *sample* of the population (not the whole thing), which is why I use n−1 for variance and standard deviation later.

## Chapter 1, Exercise 5

### Exercise 5(a) Mean
Mean describes the average of a set of numbers. It is calculated by taking the sum of the set and dividing by the number of entries.

In [2]:
import numpy as np
import pandas as pd

In [3]:
# By hand: the sum of all values divided by how many there are
mean_manual = sum(salaries) / len(salaries)
mean_manual

585690.0

In [4]:
# Using numpy.mean
np.mean(salaries)

585690.0

In [5]:
import statistics

In [6]:
# Using statistics module
statistics.mean(salaries)

585690.0

In [7]:
# Using Pandas
# Convert to series
salaries_series = pd.Series(salaries)

salaries_series.mean()

585690.0

In [8]:
from scipy import stats

In [9]:
# Using SciPy Statistics
stats.tmean(salaries)

585690.0

> **My note:** I calculated the mean from scratch first (sum ÷ count), then confirmed it against NumPy, the `statistics` module, pandas, and SciPy — they all agree.

### Exercise 5(b) Median
The median is the middle value when ordered, or in the case of an even-numbered set, the average of the two middle values.

In [10]:
# Count the values, then sort so the middle is easy to find
n = len(salaries)
ordered = sorted(salaries)

# The middle index (integer division)
mid = n // 2

# Odd n takes the middle value; even n averages the two middle values
median_manual = ordered[mid] if n % 2 else (ordered[mid-1] + ordered[mid]) / 2
median_manual

589000.0

In [11]:
# Using `statistics`
statistics.median(salaries)

589000.0

> **My note:** Used a one-line `if` to handle the odd/even split instead of writing it out longhand. Matches `statistics.median`.

### Exercise 5(c) Mode
The mode is the value (or values) that appear most often.

In [12]:
# Counter tallies how many times each value appears
from collections import Counter
counts = Counter(salaries)

# most_common(1) returns [(value, count)]; [0][0] pulls out just the value
mode_manual = counts.most_common(1)[0][0]
mode_manual

477000.0

In [13]:
# Using `statistics`
statistics.mode(salaries)

477000.0

> **My note:** I used `collections.Counter` like the hint suggested. Confirmed it against `statistics.mode`.

### Exercise 5(d) Sample Variance
Variance measures how spread out the values are from the mean. It is the average of the squared distances between each value and the mean, dividing by (n − 1) for a sample.

In [14]:
# Find the mean
mean = sum(salaries) / n

# Square the distance between each observation and the mean, then divide by (n - 1)
sample_variance_manual = sum((x - mean) ** 2 for x in salaries) / (n - 1)
sample_variance_manual

70664054444.44444

In [15]:
# Using `statistics`
statistics.variance(salaries)   # also uses n-1

70664054444.44444

> **My note:** I divide by (n − 1), not n, because the instructions say to treat this as a *sample*. Matches `statistics.variance`, which also uses n − 1.

### Exercise 5(e) Sample Standard Deviation
Standard deviation is the square root of the variance. Taking the square root puts the spread back into the same units as the original data, which makes it easier to interpret.

In [16]:
# Take the square root of the sample variance from the previous step
sample_std_manual = sample_variance_manual ** 0.5
sample_std_manual

265827.11382484

In [17]:
# Using `statistics`
statistics.stdev(salaries)

265827.11382484

> **My note:** Reused `sample_variance_manual` and took its square root, rather than recomputing from scratch. Matches `statistics.stdev`.

## Chapter 1, Exercise 6

### Exercise 6(a) Range
The range is the distance between the largest and smallest values in the set — simply the maximum minus the minimum.

In [18]:
# By hand
range_manual = max(salaries) - min(salaries)
range_manual

995000.0

> **My note:** There's no built-in statistics `range()`, so I computed it directly.

### Exercise 6(b) Coefficient of variation
The coefficient of variation is the standard deviation divided by the mean. It expresses the spread relative to the average, which makes it useful for comparing variability between data sets measured on different scales.

In [19]:
# Coefficient of variation = sample std / mean (both reused from Exercise 5)
cv = sample_std_manual / mean
cv

0.45386998894439035

> **My note:** Reused my sample std and mean from Exercise 5 instead of recomputing them.

### Exercise 6(c) Interquartile range
The interquartile range (IQR) is the spread of the middle 50% of the data — the third quartile (75th percentile) minus the first quartile (25th percentile).

In [20]:
# Grab the 25th and 75th percentiles with np.quantile, then subtract
q1, q3 = np.quantile(salaries, [0.25, 0.75])
iqr = q3 - q1
iqr

413250.0

> **My note:** I used `np.quantile` to grab Q1 and Q3 in one call, then subtracted (saving `q1`/`q3` to reuse for the QCD). `np.quantile` defaults to linear interpolation; other conventions — like the textbook's `midpoint` method — give a slightly higher IQR.

### Exercise 6(d) Quartile coefficient of dispersion
The quartile coefficient of dispersion measures relative spread using the quartiles: (Q3 − Q1) / (Q3 + Q1). Like the coefficient of variation, it is unitless, so it can compare dispersion across different data sets.

In [21]:
# Quartile coefficient of dispersion = (Q3 - Q1) / (Q3 + Q1)
qcd = (q3 - q1) / (q3 + q1)
qcd

0.338660110633067

> **My note:** Reused the `q1` and `q3` I saved back in the IQR step.

## Chapter 1, Exercise 7

### Exercise 7(a) Min-max scaling (normalizing)

In [22]:
# Find the lowest and highest values that'll map to 0 and 1
lo, hi = min(salaries), max(salaries)

# Calculate each point's relative position
min_max_scaled = [(x - lo) / (hi - lo) for x in salaries]

# Show the first five
min_max_scaled[:5]

[0.8472361809045226,
 0.7608040201005025,
 0.4221105527638191,
 0.2592964824120603,
 0.5125628140703518]

> **My note:** I did min-max scaling by hand with (x − min) / (max − min), which squeezes every value between 0 and 1.

### Exercise 7(b) Standardizing

In [23]:
# Move each point over by the mean, then scale by the standard deviation
standardized = [(x - mean) / sample_std_manual for x in salaries]

# Show the first five
standardized[:5]

[0.9717217942267801,
 0.6482032533127501,
 -0.6195380058503674,
 -1.228956652688424,
 -0.28097209094033604]

> **My note:** Standardizing by hand with (x − mean) / std, using the *sample* std from Exercise 5.

## Chapter 1, Exercise 8

The template provides code for getting both scaled versions of the data. I am commenting on the code to prove my understanding but am not using it for the calculations.

In [24]:
# Normalized (min-max) version using scikit-learn's MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

# MinMaxScaler expects a 2D array, so reshape the list into a single column
salaries_array = np.array(salaries)
salaries_2d = salaries_array.reshape(-1, 1)

# Fit the scaler, then transform the data
scaler = MinMaxScaler()
scaler.fit(salaries_2d)
scaled_salaries = scaler.transform(salaries_2d)

# Show the first 10 — the double brackets show it's still 2D
scaled_salaries[:10]

array([[0.84723618],
       [0.76080402],
       [0.42211055],
       [0.25929648],
       [0.51256281],
       [0.40603015],
       [0.78693467],
       [0.30351759],
       [0.47839196],
       [0.58492462]])

In [25]:
# Standardized (z-score) version using SciPy (stats imported earlier)
standardized_salary = stats.zscore(salaries_array)

# Show the first 10 — this one stays 1D
standardized_salary[:10]

array([ 0.97661715,  0.65146878, -0.62265912, -1.23514791, -0.28238758,
       -0.68315184,  0.74976945, -1.06879293, -0.41093461, -0.01017034])

> **My note:** `stats.zscore` uses the *population* std (ddof=0) by default, so these z-scores are slightly different from my by-hand standardized values above (which used the sample std). It doesn't change the exercise — I'm just using these as the standardized dataset that feeds the covariance/correlation steps below.

### Exercise 8(a) Covariance

In [26]:
# Template's method: statistics.covariance() — added in Python 3.10.
# The textbook environment is Python 3.8, so this raises an error.
# Per the Week 4 announcement, I show the error here, then a working method below.
try:
    print(statistics.covariance(scaled_salaries, standardized_salary))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

AttributeError: module 'statistics' has no attribute 'covariance'


> **My note:** Below I calculate the covariance with NumPy instead, using the normalized and standardized lists I built in Exercise 7.

In [27]:
# My working version — np.cov on my Exercise 7 lists (both 1D, so no need to reshape).
# np.cov returns a 2x2 matrix; the off-diagonal [0, 1] is the covariance
np.cov(min_max_scaled, standardized)[0, 1]

0.267162928467176

> **My note:** The covariance is positive, so the normalized and standardized values move in the same direction.

### Exercise 8(b) Pearson correlation coefficient

In [28]:
# Template's method: statistics.correlation() — also added in Python 3.10.
# Capture the error and use a working method below
try:
    print(statistics.correlation(scaled_salaries, standardized_salary))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

AttributeError: module 'statistics' has no attribute 'correlation'


> **My note:** I use NumPy's `np.corrcoef` below instead.

In [29]:
# My working version — np.corrcoef on the same Exercise 7 lists.
# Returns a matrix, so I grab the off-diagonal [0, 1] value.
np.corrcoef(min_max_scaled, standardized)[0, 1]

1.0

> **My note:** The correlation is 1 because the normalized and standardized data are the *same* values rescaled two ways, and scaling doesn't change the strength or direction of a relationship. `np.corrcoef` gives the exact `1.0`. Doing it by hand (like the textbook solution) can come out around `0.99` — that happens when a *population* covariance (÷n) is divided by *sample* standard deviations (÷n−1), and that ddof mismatch is where the "slightly less than 1" comes from.

# Chapter 2

In [30]:
# Load the dataset from Chapter 2
df = pd.read_csv('../ch_02/data/parsed.csv')

# Check it was loaded correctly
df.head()

,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,status,time,title,tsunami,type,types,tz,updated,url,parsed_place
0,NaN,NaN,37389218,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.008693,NaN,85.0,",ci37389218,",1.35,ml,...,automatic,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475395144,https://earthquake.usgs.gov/earthquakes/eventp...,California
1,NaN,NaN,37389202,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.020030,NaN,79.0,",ci37389202,",1.29,ml,...,automatic,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475253925,https://earthquake.usgs.gov/earthquakes/eventp...,California
2,NaN,4.4,37389194,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.021370,28.0,21.0,",ci37389194,",3.42,ml,...,automatic,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,earthquake,",dyfi,focal-mechanism,geoserve,nearby-cities,o...",-480.0,1539536756176,https://earthquake.usgs.gov/earthquakes/eventp...,California
3,NaN,NaN,37389186,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.026180,NaN,39.0,",ci37389186,",0.44,ml,...,automatic,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475196167,https://earthquake.usgs.gov/earthquakes/eventp...,California
4,NaN,NaN,73096941,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.077990,NaN,192.0,",nc73096941,",2.16,md,...,automatic,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,scit...",-480.0,1539477547926,https://earthquake.usgs.gov/earthquakes/eventp...,California


## Chapter 2, Exercise 1

Find the 95th percentile of earthquake magnitude in Japan using the mb magnitude type.

In [31]:
# Filter place by Japan, magType by 'mb', then pull 95th percentile of magnitude
df[(df.parsed_place == 'Japan') & (df.magType == 'mb')].mag.quantile(0.95)

4.9

> **My note:** Chained two filters with `&` (Japan and the `mb` magnitude type), then took the 95th percentile of `mag`.

## Chapter 2, Exercise 2

Find the percentage of earthquakes in Indonesia that were coupled with tsunamis.

In [32]:
# Filter by Indonesia, take the mean of its tsunami column, then multiply by 100
df[df.parsed_place == 'Indonesia'].tsunami.mean() * 100

23.12925170068027

> **My note:** `tsunami` is coded 0 for no tsunami and 1 for tsunami. The proportion of tsunamis is just the average of the entire column once correctly filtered to Indonesia.

## Chapter 2, Exercise 3

Calculate summary statistics for earthquakes in Nevada.

In [33]:
# Filter by Nevada, then call `.describe()`
df[df.parsed_place == 'Nevada'].describe()

,cdi,dmin,felt,gap,mag,mmi,nst,rms,sig,time,tsunami,tz,updated
count,15.000000,681.000000,15.000000,681.000000,681.000000,1.00,681.000000,681.000000,681.000000,6.810000e+02,681.0,681.0,6.810000e+02
mean,2.440000,0.166199,2.400000,153.668120,0.500073,2.84,12.618209,0.151986,10.970631,1.538314e+12,0.0,-480.0,1.538402e+12
std,0.501142,0.166228,4.626013,68.735302,0.696710,NaN,9.866963,0.084662,19.607150,5.965637e+08,0.0,0.0,6.010951e+08
min,2.000000,0.001000,1.000000,29.140000,-0.500000,2.84,3.000000,0.000500,0.000000,1.537247e+12,0.0,-480.0,1.537307e+12
25%,2.000000,0.053000,1.000000,97.380000,-0.100000,2.84,6.000000,0.106900,0.000000,1.537854e+12,0.0,-480.0,1.537928e+12
50%,2.200000,0.112000,1.000000,149.140000,0.400000,2.84,10.000000,0.146300,2.000000,1.538280e+12,0.0,-480.0,1.538428e+12
75%,2.900000,0.233000,1.000000,199.720000,0.900000,2.84,16.000000,0.187100,12.000000,1.538821e+12,0.0,-480.0,1.538878e+12
max,3.300000,1.414000,19.000000,355.910000,2.900000,2.84,61.000000,0.863400,129.000000,1.539461e+12,0.0,-480.0,1.539483e+12


> **My note:** `describe()` quickly pulls every summary stat (count, mean, std, min, quartiles, max) for all numeric columns at once.

## Chapter 2, Exercise 4

Add a column indicating whether the earthquake happened in a country or US state that is on the Ring of Fire.

In [34]:
# Easiest to use a string, because then we can just call `str.contains()`
ring_of_fire = (
    'Alaska|Antarctic|Bolivia|California|Canada|Chile|Costa Rica|Ecuador|'
    'Fiji|Guatemala|Indonesia|Japan|Kermadec Islands|Mexico|New Zealand|'
    'Peru|Philippines|Russia|Taiwan|Tonga|Washington'
)

# Create the column: True if the place is on the list...
df['ring_of_fire'] = (
    df.parsed_place.str.contains(ring_of_fire)

    # ...but exclude New Mexico, which 'Mexico' would otherwise match
    & ~df.parsed_place.str.contains('New Mexico')
)

# Check first five rows
df[['parsed_place', 'ring_of_fire']].head()

,parsed_place,ring_of_fire
0,California,True
1,California,True
2,California,True
3,California,True
4,California,True


> **My note:** 'Mexico' also matches 'New Mexico', so I excluded it with `& ~...str.contains('New Mexico')`. I also searched for 'Antarctic' instead of 'Antarctica', per the instructions.

## Chapter 2, Exercise 5

Calculate the number of earthquakes in the Ring of Fire locations and the number outside of them.

In [35]:
# Count True (on the Ring of Fire) vs False (outside it)
df.ring_of_fire.value_counts()

True     7188
False    2144
Name: ring_of_fire, dtype: int64

> **My note:** Most rows come back True, which makes sense — the data is mostly California quakes, and California is on the Ring of Fire.

## Chapter 2, Exercise 6

Find the tsunami count along the Ring of Fire.

In [36]:
# Filter by ring_of_fire, then count tsunamis
df[df.ring_of_fire].tsunami.sum()

45

> **My note:** Since tsunamis are coded as 0 for no and 1 for yes, we can use `sum()` to count them.